In [ ]:
from option_finder import *
from option_data_plotter import *

In [ ]:
try:
    logger
except NameError:
    logger = get_rotating_logger("jupyter", f'logs/all_options.log')

chain_dir = 'chain'
quotes_dir = 'quotes'
data_dir = 'data'
cookie_file = 'cookie.txt'
self = OptionFinder(logger, chain_dir=chain_dir, report_dir=data_dir)

### If immediate data refresh is needed, run this

In [ ]:
age_dict = dict([(os.path.basename(f), time.time() - os.path.getmtime(f)) for f in glob(os.path.join(self.chain_dir, '*'))])
symlist = sorted([k for k, v in age_dict.items() if v <= 3600], key=age_dict.get)
print(f'{len(symlist)} symbols, age: {age_dict[symlist[0]]:.01f}" {age_dict[symlist[-1]]:.01f}", {" ".join(symlist)}')

### Read downloaded data, check the data age chart to make sure the data are fresh

In [ ]:
df_quotes, df_shortint, df_vola = self.get_quote_df(symlist)
_t0 = time.time()
_df = self.build_option_df(symlist)
_t1 = time.time()
print(f'build_option_df {_t1 - _t0:.1f} seconds')
dfcp = self.concat_put_call_options(_df)
dfcp = bucketize_dte(add_moneyness_columns(dfcp))
_t2 = time.time()
print(f'concat_put_call_options {_t2 - _t1:.1f} seconds')
px.bar(check_data_age(_df), y=['load_age', 'quote_age'], barmode='group', title=f"Data Ages", width=800, height=300).show()
print(f'px.bar {time.time() - _t2:.1f} seconds')
dfcp.loc[:, ['dte', 'expDt']].groupby('dte').first().head(24).tail(20).T

In [ ]:
try:
    d2e = count_days_from_earning_reports(df_quotes)['earningDays'].to_dict()
except KeyError:
    d2e = {}
print('Days to E:', d2e)

### ImpVola Overview

In [ ]:
plot_iv_statistics(dfcp)

## Put Options
Ignore no-bid or low open interest (minimum open interests is 100)

In [ ]:
_lodf = [compute_all_time_decay_metrics_for_symbol(dfcp, symbol, 'P', ignore_no_bid=True, oi_lb=100) for symbol in symlist]
_dfp = pd.concat(_lodf)
_dfp['dthr'] = _dfp.dth/_dfp.dte
_dfp['dtzr'] = _dfp.dtz/_dfp.dte
_index = ['symbol', 'dte', 'strike']
add_cols = ['Delta', 'pctSpread', 'lastPrice', 'Theta', 'moneyness', 'expDt', 'OpenInterest']
dfp = _dfp.set_index(_index).join(dfcp[dfcp.type=='P'].set_index(_index).loc[:, add_cols]).reset_index()
dfp['pctProfit'] = dfp.premium/2/dfp.strike/dfp.dth*100*365
dfp['E'] = dfp.symbol.apply(d2e.get)
print('hdte_resid check:', dfp[(dfp.hdte_resid < dfp.resid) & (np.abs(dfp.hdte_resid - dfp.resid) >= 1e-6)].shape)

### Put options with no earning date on or before expiration date

In [ ]:
hdte_resid_ub = 0.6
spread_ub = 17
_filter = (dfp.moneyness <= 0.99) & (dfp.pctSpread <= spread_ub) & (dfp.hdte_resid<=hdte_resid_ub) & (dfp.E.isna() |(dfp.E > dfp.dte))
_filter = _filter & ~dfp.symbol.str.contains('META|TSLA|MSFT')
_dfp = dfp[_filter].sort_values(by='pctProfit', ascending=False)
print(_dfp.shape)
_dfp.head(20)

### Put options including stocks near earning dates

In [ ]:
_moneyness_ub = 0.95
_hdte_resid_ub = 0.7
_filter = (dfp.moneyness <= _moneyness_ub) & (dfp.pctSpread <= 5) & (dfp.hdte_resid<=_hdte_resid_ub)
_dfp = dfp[_filter].sort_values(by='pctProfit', ascending=False)
print(_dfp.shape)
_dfp.head(20)

### Put options for specific symbols

In [ ]:
_filter = dfp.symbol.str.contains('QQQ') & (dfp.moneyness <= 1) & (dfp.pctProfit >= 60) #& (dfp.dte < dfp.E)
#_filter = _filter & (dfp.premium >= 3.1) #& (dfp.hdte_resid<=0.8)
_dfp = dfp[_filter].sort_values(by='pctProfit', ascending=False)
print(_dfp.shape)
_dfp.head(20)

### Put options: top 500 in-the-money

In [ ]:
px.scatter(dfp[dfp.moneyness <= 1].sort_values(by='pctProfit', ascending=False).head(500), x='hdte_resid', y='pctProfit', color='symbol', height=600)

### Call Options: ignore no-bid or low open interest (minimum open interests is 100)

In [ ]:
_lodfc = [compute_all_time_decay_metrics_for_symbol(dfcp, symbol, 'C', ignore_no_bid=True, oi_lb=100) for symbol in symlist]
_dfc = pd.concat(_lodfc)
_dfc['dthr'] = _dfc.dth/_dfc.dte
_dfc['dtzr'] = _dfc.dtz/_dfc.dte
_index = ['symbol', 'dte', 'strike']
add_cols = ['lastPrice', 'Delta', 'Theta', 'expDt', 'OpenInterest', 'pctSpread', 'overpaid', 'leverage']
dfc = _dfc.set_index(_index).join(dfcp[dfcp.type=='C'].set_index(_index).loc[:, add_cols]).reset_index()
dfc['E'] = dfc.symbol.apply(d2e.get)
print('hdte_resid check:', dfc[(dfc.hdte_resid < dfc.resid) & (np.abs(dfc.hdte_resid - dfc.resid) >= 1e-6)].shape)

In [ ]:
_filter = (dfc.symbol != 'TLT') & (dfc.dte >= 90) & (dfc.hdte_resid >= 0.95) & (dfc.overpaid <= 0.05) & (dfc.pctSpread <= 2)
_dfc = dfc[_filter].drop(columns=['dth', 'dtz', 'dthr', 'dtzr']).sort_values(by='leverage', ascending=False)
_dfc.head(20)

In [ ]:
_filter = (dfc.pctSpread <= 10) & (dfc.strike <= dfc.lastPrice) & (dfc.dte >= 60)
px.scatter(dfc[_filter].sort_values(by='leverage', ascending=False).head(1000), x='hdte_resid', y='leverage', color='symbol', height=600)

In [ ]:
dfc[(dfc.symbol=='SPY') & (dfc.dte >= 90) & (dfc.strike <= dfc.lastPrice) & (dfc.leverage <= 8)].sort_values(by='leverage', ascending=False).head(20)

### Put Debit Spread

In [ ]:
def calc_pds_debit(df):
    strike0 = df.strike.iloc[0]
    spreads = list(df.strike.iloc[[1, -1]] - strike0)
    return int((df.mid.iloc[0]*2 + df.mid.iloc[1] - df.mid.iloc[-1])*100)/100, strike0, spreads

def get_final_dfpds(dfpds):
    pds_list = []
    for idx in dfpds.index:
        _df = put_debit_spread(dfpds, *idx, dfcp)
        debit, strike0, spreads = calc_pds_debit(_df)
        pds_list.append({'symbol': idx[0], 'dte': idx[1], 'debit': debit, 'strike0': strike0, 'spread1': spreads[0], 'spread2': spreads[1]})
    return dfpds.join(pd.DataFrame(pds_list, index=dfpds.index))

In [ ]:
dfpds = select_pds_deltas(dfcp, 35, 45)
dfpds

## Total open interests and volumes for all dte and strikes

In [ ]:
_df = dfcp.loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False)

### It may be of interest to look at OpenInterests and Volumes for 8-weeks and 1-year DTE clusters

In [ ]:
for _dte_cluster in ['8wk', '1yr']:
    _df = dfcp[dfcp.dte_cluster==_dte_cluster].loc[:, ['symbol', 'type', 'OpenInterest', 'Volume']].groupby(['symbol', 'type']).sum().reset_index()
    plot_metrics_in_one_row(_df, ['symbol', 'type'], ['OpenInterest', 'Volume'], shared_y=False, log_y_threshold=500, horizontal_spacing=0.03)

In [ ]:
_df = calc_overall_put_call_ratios(dfcp)
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_df = calc_overall_put_call_ratios(dfcp[dfcp.cluster=='atm'])
_df = _df.rename(columns=dict([(c, 'atm '+c) for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:])

In [ ]:
_dte_lb = 90
_dte_ub = 180
_df = calc_overall_put_call_ratios(dfcp[(dfcp.cluster=='atm') & (dfcp.dte >= _dte_lb) & (dfcp.dte <= _dte_ub)])
_df = _df.rename(columns=dict([(c, f'atm dte {_dte_lb} to {_dte_ub} {c}') for c in _df.columns[1:]]))
plot_metrics_in_one_row(_df, ['symbol'], list(_df.columns)[-2:], shared_y=False)

### The End